# Point Cloud Processing: Pemisahan Lantai dan Dinding

Segmentasi bidang dari data LiDAR sintetis menggunakan dua metode:
1. **RANSAC iteratif** — plane fitting berbasis konsensus geometrik
2. **Normal Vector Clustering + DBSCAN** — clustering berbasis arah normal permukaan

| Komponen | Keterangan |
|---|---|
| Lantai | Bidang horizontal, z = 0 |
| Dinding A | Bidang vertikal, y = 0 |
| Dinding B | Bidang vertikal, x = 0 (tegak lurus dinding A) |

## 0. Setup

In [ ]:
# !pip install open3d  # uncomment jika belum terinstal

In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
import time

SEED = 42
rng = np.random.default_rng(SEED)

# Palet warna konsisten untuk semua plot
LABEL_COLORS = {
    0:  '#1f77b4',  # lantai  → biru
    1:  '#ff7f0e',  # dinding A → oranye
    2:  '#2ca02c',  # dinding B → hijau
    -1: '#aaaaaa',  # unclassified → abu
}
LABEL_NAMES = {0: 'Lantai', 1: 'Dinding A', 2: 'Dinding B', -1: 'Unclassified'}

print(f"NumPy  : {np.__version__}")
print(f"Open3D : {o3d.__version__}")